rmse + log = 96.24

In [5]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_percentage_error
import re

all_data = pd.read_csv('/kaggle/input/datasets/levkremlev/prepared-features05-05/prepared_features (1).csv')

if 'РТО' in all_data.columns:
    all_data = all_data.rename(columns={'РТО': 'rto'})
elif 'RTO' in all_data.columns:
    all_data = all_data.rename(columns={'RTO': 'rto'})

all_data = all_data.sort_values(['new_id', 'Месяц'])

for i in range(1, 10):
    all_data[f'rto_lag_{i}'] = all_data.groupby('new_id')['rto'].shift(i)

all_data['rto_mean_3'] = all_data[[f'rto_lag_{i}' for i in [1, 2, 3]]].mean(axis=1)
all_data['rto_mean_6'] = all_data[[f'rto_lag_{i}' for i in range(1, 7)]].mean(axis=1)

all_data['month_sin'] = np.sin(2 * np.pi * all_data['Месяц'] / 12)
all_data['month_cos'] = np.cos(2 * np.pi * all_data['Месяц'] / 12)

def extract_num(x):
    res = re.findall(r"[-+]?\d*\.\d+|\d+", str(x))
    return float(res[0]) if res else np.nan

all_data['area_numeric'] = all_data['Торговая площадь, категориальный'].apply(extract_num)
all_data['area_numeric'] = all_data['area_numeric'].fillna(all_data['area_numeric'].median())

# Эффективность и тренды
all_data['rto_per_area'] = all_data['rto_lag_1'] / (all_data['area_numeric'] + 1e-6)
all_data['rto_per_kassa'] = all_data['rto_lag_1'] / (all_data['Количество касс'] + 1e-6)
all_data['trend_1_2'] = all_data['rto_lag_1'] / (all_data['rto_lag_2'] + 1e-6)
all_data['rto_vs_mean6'] = all_data['rto_lag_1'] / (all_data['rto_mean_6'] + 1e-6)
all_data['kassa_per_area'] = all_data['Количество касс'] / (all_data['area_numeric'] + 1e-6)

region_stats = all_data.groupby('Регион')['rto_lag_1'].transform('mean')
all_data['region_rto_mean'] = region_stats

features = [
    'Населенный пункт', 'Регион', 'area_numeric',
    'rto_lag_1', 'rto_lag_2', 'rto_lag_3', 'rto_lag_6', 'rto_lag_9',
    'month_sin', 'month_cos', 'trend_1_2',
    'rto_per_kassa', 'Численность населения', 'Количество касс',
    'Трафик пеший, в час', 'Пятерочки (500 м)',
    'rto_mean_3', 'rto_mean_6', 'rto_per_area', 
    'rto_vs_mean6', 'kassa_per_area', 'region_rto_mean'
]
cat_features = ['Населенный пункт', 'Регион']

train_df = all_data[all_data['Месяц'] == 10].copy()
X = train_df[features]
y_log = np.log1p(train_df['rto']) 

# --- 4. ПОДГОТОВКА ТЕСТА ДЛЯ НОЯБРЯ ---
test_df = all_data[all_data['Месяц'] == 10].copy()

for i in range(9, 1, -1): # Сдвигаем с конца: 9-й становится 8-м и т.д.
    test_df[f'rto_lag_{i}'] = test_df[f'rto_lag_{i-1}']
test_df['rto_lag_1'] = test_df['rto'] # Октябрь становится первым лагом для Ноября

# 2. Пересчитываем вычисляемые признаки для Ноября
test_df['rto_mean_3'] = test_df[[f'rto_lag_{i}' for i in [1, 2, 3]]].mean(axis=1)
test_df['rto_mean_6'] = test_df[[f'rto_lag_{i}' for i in range(1, 7)]].mean(axis=1)
test_df['rto_per_area'] = test_df['rto_lag_1'] / (test_df['area_numeric'] + 1e-6)
test_df['rto_vs_mean6'] = test_df['rto_lag_1'] / (test_df['rto_mean_6'] + 1e-6)
test_df['rto_per_kassa'] = test_df['rto_lag_1'] / (test_df['Количество касс'] + 1e-6)
test_df['trend_1_2'] = test_df['rto_lag_1'] / (test_df['rto_lag_2'] + 1e-6)

# Обновляем время на Ноябрь
test_df['month_sin'] = np.sin(2 * np.pi * 11 / 12)
test_df['month_cos'] = np.cos(2 * np.pi * 11 / 12)

X_test_final = test_df[features]

n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
oof_preds_log = np.zeros(len(X))
test_preds_log = np.zeros(len(X_test_final))

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y_log)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y_log.iloc[train_idx], y_log.iloc[val_idx]
    
    model = CatBoostRegressor(iterations=2000, learning_rate=0.03, depth=5,l2_leaf_reg=10, 
                              loss_function='RMSE', random_seed=42, verbose=500)
    
    model.fit(X_tr, y_tr, cat_features=cat_features, eval_set=(X_val, y_val), early_stopping_rounds=100)
    
    oof_preds_log[val_idx] = model.predict(X_val)
    test_preds_log += model.predict(X_test_final) / n_splits

final_oof_preds = np.expm1(oof_preds_log)
final_test_preds = np.expm1(test_preds_log)

total_mape = mean_absolute_percentage_error(train_df['rto'], final_oof_preds) * 100
print(f"\nИтоговый MAPE: {total_mape:.2f}%")
print(f"Баллы: {100 - min(total_mape, 100):.2f}")

submission = pd.DataFrame({'new_id': test_df['new_id'], 'rto': final_test_preds})
submission.to_csv('test.csv', index=False)

0:	learn: 0.4508216	test: 0.4481725	best: 0.4481725 (0)	total: 9.53ms	remaining: 19s
500:	learn: 0.0567528	test: 0.0614755	best: 0.0614755 (500)	total: 3.62s	remaining: 10.8s
1000:	learn: 0.0537697	test: 0.0601849	best: 0.0601849 (1000)	total: 6.75s	remaining: 6.74s
1500:	learn: 0.0517296	test: 0.0596533	best: 0.0596533 (1500)	total: 10s	remaining: 3.34s
1999:	learn: 0.0499382	test: 0.0594697	best: 0.0594658 (1993)	total: 13.3s	remaining: 0us

bestTest = 0.05946581344
bestIteration = 1993

Shrink model to first 1994 iterations.
0:	learn: 0.4498642	test: 0.4520004	best: 0.4520004 (0)	total: 9.24ms	remaining: 18.5s
500:	learn: 0.0549123	test: 0.0694015	best: 0.0694015 (500)	total: 3.56s	remaining: 10.6s
1000:	learn: 0.0523347	test: 0.0682728	best: 0.0682728 (1000)	total: 6.88s	remaining: 6.87s
1500:	learn: 0.0508981	test: 0.0677302	best: 0.0677299 (1499)	total: 9.95s	remaining: 3.31s
1999:	learn: 0.0495902	test: 0.0673092	best: 0.0673092 (1999)	total: 13.2s	remaining: 0us

bestTest = 0.0

новый 96.29

In [6]:
from IPython.display import FileLink
FileLink('test.csv')

/kaggle/working/test.csv